# ISIC 2018 Task 1 - Exploratory Data Analysis

This notebook explores the ISIC 2018 Task 1 lesion-segmentation dataset before any
modelling. It produces three figures that motivate the preprocessing choices and are
reported in Chapter 5:

1. A sample grid of dermoscopic images with their ground-truth masks and an overlay.
2. The distribution of the lesion-area-to-image ratio (how much of each image is lesion).
3. The distribution of native image resolutions across the dataset.

Figures are saved to `results/figures/eda/`. The raw images are never modified.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

sns.set_theme(style="whitegrid", context="notebook")


def find_root(start: Path) -> Path:
    """Walk up from `start` until the folder containing data/raw is found."""
    for p in [start, *start.parents]:
        if (p / "data" / "raw").exists():
            return p
    raise RuntimeError("could not locate the repository root (no data/raw found)")


ROOT = find_root(Path.cwd())
RAW = ROOT / "data" / "raw"
FIG_DIR = ROOT / "results" / "figures" / "eda"
FIG_DIR.mkdir(parents=True, exist_ok=True)

SPLITS = ("train", "val", "test")
RNG = np.random.default_rng(42)  # fixed seed so the sample grid is reproducible

print("repository root:", ROOT)
print("figures ->", FIG_DIR)

In [ ]:
def list_pairs(split: str):
    """Return sorted (image_path, mask_path) pairs for a split."""
    img_dir = RAW / split / "images"
    msk_dir = RAW / split / "masks"
    images = sorted(p for p in img_dir.glob("*.jpg"))
    pairs = []
    for img in images:
        mask = msk_dir / f"{img.stem}_segmentation.png"
        if mask.exists():
            pairs.append((img, mask))
    return pairs


pairs = {s: list_pairs(s) for s in SPLITS}
for s in SPLITS:
    print(f"{s:5s}: {len(pairs[s])} image/mask pairs")

## 1. Sample images and masks

A random, seeded selection of training samples. Each row shows the dermoscopic image, its
binary ground-truth mask, and the mask drawn as a red overlay on the image. This gives a
qualitative feel for lesion appearance, border sharpness, colour variation, and artefacts
(hair, rulers, ink marks).

In [ ]:
n_samples = 5
sample_idx = RNG.choice(len(pairs["train"]), size=n_samples, replace=False)

fig, axes = plt.subplots(n_samples, 3, figsize=(9, 3 * n_samples))
col_titles = ["Dermoscopic image", "Ground-truth mask", "Overlay"]

for row, idx in enumerate(sample_idx):
    img_path, mask_path = pairs["train"][idx]
    image = Image.open(img_path).convert("RGB")
    mask = Image.open(mask_path).convert("L")
    img_arr = np.asarray(image)
    mask_arr = np.asarray(mask) > 127  # boolean lesion map

    # Red overlay where the mask is positive.
    overlay = img_arr.copy()
    overlay[mask_arr] = (0.5 * overlay[mask_arr] + 0.5 * np.array([255, 0, 0])).astype(np.uint8)

    for col, panel in enumerate([img_arr, mask_arr, overlay]):
        ax = axes[row, col]
        ax.imshow(panel, cmap="gray" if col == 1 else None)
        ax.set_xticks([])
        ax.set_yticks([])
        if row == 0:
            ax.set_title(col_titles[col])
    axes[row, 0].set_ylabel(img_path.stem, fontsize=8)

fig.suptitle("ISIC 2018 Task 1 - sample images, masks, and overlays (train split)", y=1.01)
fig.tight_layout()
out = FIG_DIR / "sample_grid.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print("saved", out)
plt.show()

## 2. Lesion-area-to-image ratio

For every training mask, the lesion-area ratio is the fraction of pixels labelled lesion:
`ratio = (positive pixels) / (total pixels)`. The distribution shows how much of a typical
image the lesion occupies. A strong skew toward small ratios indicates class imbalance
(far more background than lesion pixels), which justifies a region-sensitive loss such as
Dice combined with BCE rather than BCE alone.

In [ ]:
ratios = []
for _img, mask_path in pairs["train"]:
    mask = np.asarray(Image.open(mask_path).convert("L")) > 127
    ratios.append(mask.mean())  # fraction of lesion pixels
ratios = np.array(ratios)

print(f"lesion-area ratio over {len(ratios)} training masks:")
print(f"  min    {ratios.min():.4f}")
print(f"  max    {ratios.max():.4f}")
print(f"  mean   {ratios.mean():.4f}")
print(f"  median {np.median(ratios):.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(ratios, bins=40, ax=ax, color="#4c72b0")
ax.axvline(ratios.mean(), color="#c44e52", linestyle="--", label=f"mean = {ratios.mean():.3f}")
ax.axvline(np.median(ratios), color="#55a868", linestyle=":", label=f"median = {np.median(ratios):.3f}")
ax.set_xlabel("Lesion area / image area")
ax.set_ylabel("Number of images")
ax.set_title("Lesion-area-to-image ratio (train split)")
ax.legend()
fig.tight_layout()
out = FIG_DIR / "lesion_area_ratio_hist.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print("saved", out)
plt.show()

## 3. Resolution distribution

Native image dimensions across all three splits. Reading only the image headers (not the
pixels) keeps this fast. A wide spread of resolutions is the direct justification for
resizing every image to a single fixed size (224x224) before training: the models need a
consistent input, and 224x224 is additionally required by Swin-UNet.

In [ ]:
widths, heights = [], []
for s in SPLITS:
    for img_path, _mask in pairs[s]:
        with Image.open(img_path) as im:
            w, h = im.size
        widths.append(w)
        heights.append(h)
widths = np.array(widths)
heights = np.array(heights)
megapixels = (widths * heights) / 1e6

print(f"resolution over {len(widths)} images (all splits):")
print(f"  width  : min {widths.min()}  max {widths.max()}  mean {widths.mean():.0f}")
print(f"  height : min {heights.min()}  max {heights.max()}  mean {heights.mean():.0f}")
print(f"  distinct (w,h) sizes: {len(set(zip(widths.tolist(), heights.tolist())))}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
sns.histplot(widths, bins=30, ax=axes[0], color="#4c72b0")
axes[0].set_title("Image width (px)")
axes[0].set_xlabel("width")
sns.histplot(heights, bins=30, ax=axes[1], color="#dd8452")
axes[1].set_title("Image height (px)")
axes[1].set_xlabel("height")
axes[2].scatter(widths, heights, s=6, alpha=0.3, color="#55a868")
axes[2].set_title("Width vs height")
axes[2].set_xlabel("width")
axes[2].set_ylabel("height")
fig.suptitle("ISIC 2018 Task 1 - native resolution distribution (all splits)")
fig.tight_layout()
out = FIG_DIR / "resolution_distribution.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print("saved", out)
plt.show()

## Summary

- The dataset contains dermoscopic images with binary lesion masks across train/val/test.
- Lesions occupy a wide but typically small fraction of each image, motivating a
  Dice + BCE loss for the class imbalance.
- Native resolutions vary substantially, justifying a fixed 224x224 input for all models.

All figures are saved under `results/figures/eda/` for use in the report.